In [ ]:
import kaggle_benchmarks as kbench
import textwrap

@kbench.task(name="executive_function_test")
def executive_function_test(llm):
    prompts = [
        # 1. Logic Puzzle
        "There are three boxes, one contains only apples, one contains only oranges, and one contains both apples and oranges. The boxes have been incorrectly labeled such that no label is correct. You are allowed to draw one fruit from one box of your choosing. Which box would you choose and why, to correctly label all boxes?",
        # 2. Planning Task
        "Plan a 3-day weekend trip to a major city you know well. Your budget is $500 after travel and accommodation. Outline the itinerary day-by-day, including activities, meals, and estimated costs for each, ensuring you stay within budget.",
        # 3. Overriding Habit (Cognitive Reflection Test)
        "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Explain your reasoning step-by-step.",
        # 4. Sequential Instruction
        "Provide a step-by-step guide on how to assemble a simple wooden bookshelf that has a top, a bottom, two sides, and a back panel. Assume all screws and a screwdriver are provided.",
        # 5. Re-ordering a Process
        "Here are the steps to bake a cake, but they are out of order: 1. Pour batter into a pan. 2. Mix dry ingredients. 3. Bake in the oven. 4. Pre-heat the oven. 5. Mix wet ingredients with dry ingredients. Please list the steps in the correct logical sequence and explain why the order matters.",
        # 6. Goal-driven Creation
        "Write a short story about a detective who must solve a case with three clues: a dropped ticket stub, a single feather, and a cryptic note that says 'The bird flies at midnight.' The story must have a clear beginning, middle, and end.",
        # 7. Complex System Explanation
        "Explain the water cycle to a 10-year-old. You must include the concepts of evaporation, condensation, precipitation, and collection, and describe how they form a continuous loop.",
        # 8. Strategic Thinking
        "You are playing Tic-Tac-Toe. It is your turn. The board is: X O _ | _ X _ | O _ _ . What is your best move and what is your strategy for the rest of the game?",
        # 9. Debugging/Problem-Solving
        "A user reports: 'My computer is running very slow.' List a sequence of at least 5 diagnostic questions you would ask them to identify the root cause of the problem.",
        # 10. Multi-constraint Task
        "Create a healthy, balanced meal plan for one day (breakfast, lunch, dinner). The plan must be vegetarian, contain at least 80 grams of protein in total, and must not use any soy-based products. List the meals and their approximate protein content."
    ]

    criteria = [
        "The response breaks down the problem into logical, sequential steps.",
        "The response demonstrates a clear plan to get from the problem to the solution.",
        "The response correctly addresses all constraints and conditions mentioned in the prompt.",
        "The response avoids common, simplistic answers and shows deeper reasoning (overrides cognitive 'habits')."
    ]

    for i, prompt in enumerate(prompts):
        response = llm.prompt(prompt)

        assessment = kbench.assertions.assess_response_with_judge(
            criteria=criteria,
            response_text=response,
            judge_llm=kbench.judge_llm
        )

        if assessment is None:
            kbench.assertions.assert_fail(
                expectation=f"Judge LLM failed to return a valid assessment for prompt {i+1}."
            )
            continue

        for result in assessment.results:
            kbench.assertions.assert_true(
                result.passed,
                expectation=textwrap.dedent(f"""
                    Executive function test failed on prompt {i+1}.
                    Prompt: '{prompt[:80]}...'
                    Criterion: '{result.criterion}'
                    Reason: {result.reason}
                """)
            )

executive_function_test.run(kbench.llm)